In [ ]:
book_name = "book_name"

In [14]:
def imposition(n: int, pages_per_signature: int = 20):
    """
    Return (fronts, backs) for printing a book of n pages.
    pages_per_signature should be a multiple of 4 (default 20 = 5 sheets).
    The last signature is padded with None to a multiple of 4.
    """
    fronts = []
    backs = []
    start = 1

    while start <= n:
        remaining = n - start + 1
        size = min(pages_per_signature, remaining)
        # Pad to next multiple of 4
        padded_size = ((size + 3) // 4) * 4
        pages = list(range(start, start + size)) + [None] * (padded_size - size)

        # Build sheets for this signature
        for i in range(padded_size // 4):
            # local 1-based indices mapped to 0-based list
            fl = pages[padded_size - 2*i - 1]   # front left
            fr = pages[1 + 2*i - 1]             # front right
            bl = pages[2 + 2*i - 1]             # back left
            br = pages[padded_size - 1 - 2*i - 1]  # back right

            fronts.extend([fl, fr])
            backs.extend([bl, br])

        start += size

    return fronts, backs

In [15]:
import pymupdf

src = pymupdf.open(f"{book_name}.pdf")
n = len(src)

front_doc = pymupdf.open()
back_doc  = pymupdf.open()

# --- same rectangle setup as your code --- # -> pymupdf 4-up rect setup
width, height = pymupdf.paper_size("a4")
r = pymupdf.Rect(0, 0, width, height)
r1 = r / 2
r2 = r1 + (r1.width, 0, r1.width, 0)
r3 = r1 + (0, r1.height, 0, r1.height)
r4 = pymupdf.Rect(r1.br, r.br)
r_tab = [r1, r2, r3, r4]        # TL, TR, BL, BR

front, back = imposition(n)

def make_doc(doc, index):
    k=0
    page = doc.new_page(-1, width=width, height=height)
    for pno in index:
        if k == 4:
            k=0
            page = doc.new_page(-1, width=width, height=height)
        if pno == None:
            k+=1
            continue
        page.show_pdf_page(r_tab[k % 4], src, pno-1)
        k+=1

make_doc(front_doc, front)
make_doc(back_doc, back)

front_doc.save("front.pdf", garbage=3, deflate=True)
back_doc .save("back.pdf",  garbage=3, deflate=True)

def shuffle_join(pdf_a, pdf_b, output):
    a = pymupdf.open(pdf_a)
    b = pymupdf.open(pdf_b)
    out = pymupdf.open()

    # interleave: A1, B1, A2, B2, ...
    for i in range(max(len(a), len(b))):
        if i < len(a):
            out.insert_pdf(a, from_page=i, to_page=i)
        if i < len(b):
            out.insert_pdf(b, from_page=i, to_page=i)

    out.save(output, garbage=3, deflate=True)

shuffle_join("front.pdf", "back.pdf", f"[final] {book_name}.pdf")
